# 11 · 처짐과 균열 — KDS 14 20 30

| 항목 | 조문 |
|---|---|
| 최소 두께 (처짐 계산 생략) | KDS 14 20 30 표 4.2-1 |
| 유효단면2차모멘트 (Branson) | KDS 14 20 30 식 (4.2-1) |
| 장기처짐 계수 $\lambda_\Delta = \xi/(1+50\rho')$ | KDS 14 20 30 식 (4.2-4) |
| 최대 허용처짐 | KDS 14 20 30 표 4.2-2 |
| 균열 제어 철근 간격 | KDS 14 20 20 4.2.3(4) |
| 수축·온도철근 | KDS 14 20 50 4.6.2 |

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np

# 한글 글꼴이 없는 환경에서도 그림이 깨지지 않도록 축 라벨은 ASCII 로 둔다
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 96

In [ ]:
from concreteproperties import ConcreteSection
from sectionproperties.pre.library import concrete_rectangular_section

from concreteproperties_kds import KDS


def beam_section(fck=27, fy=400):
    """400 x 600 보 단면 (상부 2-D16, 하부 4-D22, 피복 50 mm)."""
    kds = KDS(column_type="tie")
    conc = kds.create_concrete_material(compressive_strength=fck)
    steel = kds.create_steel_material(yield_strength=fy)

    geom = concrete_rectangular_section(
        d=600, b=400,
        dia_top=16, area_top=198.6, n_top=2, c_top=50,
        dia_bot=22, area_bot=387.1, n_bot=4, c_bot=50,
        n_circle=16, conc_mat=conc, steel_mat=steel,
    )
    conc_sec = ConcreteSection(geom)
    kds.assign_concrete_section(conc_sec)
    return kds, conc_sec


def column_section(fck=27, fy=400, column_type="tie"):
    """500 x 500 기둥 단면 (8-D22, 피복 50 mm)."""
    kds = KDS(column_type=column_type)
    conc = kds.create_concrete_material(compressive_strength=fck)
    steel = kds.create_steel_material(yield_strength=fy)

    geom = concrete_rectangular_section(
        d=500, b=500,
        dia_top=22, area_top=387.1, n_top=3, c_top=50,
        dia_bot=22, area_bot=387.1, n_bot=3, c_bot=50,
        dia_side=22, area_side=387.1, n_side=1, c_side=50,
        n_circle=16, conc_mat=conc, steel_mat=steel,
    )
    conc_sec = ConcreteSection(geom)
    kds.assign_concrete_section(conc_sec)
    return kds, conc_sec

In [ ]:
from concreteproperties_kds.serviceability import (
    check_crack_control,
    check_deflection,
    long_term_deflection_factor,
    minimum_thickness,
    shrinkage_temperature_reinforcement,
    shrinkage_temperature_spacing,
)

SPAN, FY = 8000.0, 400.0

kds, conc_sec = beam_section()
conc = conc_sec.concrete_geometries[0].material

gross = kds.get_transformed_gross_properties(
    elastic_modulus=conc.elastic_modulus
)
cracked = kds.calculate_cracked_properties(theta=0)
cracked.calculate_transformed_properties(
    elastic_modulus=conc.elastic_modulus
)

h_min = minimum_thickness(span=SPAN, member="보", support="단순지지", fy=FY)
print(f"최소 두께  l/16 = {h_min:.1f} mm,  h = 600.0 mm"
      f"  ->  {'생략 가능' if h_min <= 600 else '처짐 계산 필요'}")

## 허용처짐은 조건마다 비교 대상이 다르다

KDS 14 20 30 표 4.2-2 는 조건마다 **비교하는 처짐의 종류**를 달리 정한다.
이 점을 놓치면 검토가 과도하게 보수적이 된다.

| 조건 | 허용처짐 | 비교 대상 |
|---|---|---|
| 지붕, 비구조재 없음 | $l/180$ | 활하중 즉시처짐 |
| 바닥, 비구조재 없음 | $l/360$ | 활하중 즉시처짐 |
| 손상되기 쉬운 비구조재 | $l/480$ | 부착 후 발생 처짐 |
| 손상되지 않는 비구조재 | $l/240$ | 부착 후 발생 처짐 |

In [ ]:
for condition in [
    "바닥_비구조재없음", "손상되기쉬운_비구조재", "손상되지않는_비구조재",
]:
    res = check_deflection(
        span=SPAN, m_sustained=120e6, m_live=60e6,
        m_cr=cracked.m_cr, i_g=gross.ixx_c, i_cr=cracked.ixx_c_cr,
        e_c=conc.elastic_modulus,
        rho_prime=2 * 198.6 / (400 * 550),
        duration="5년이상", condition=condition,
    )
    res.print_results()
    print()

전체 처짐 34.4 mm 를 모든 한계와 비교하면 세 조건 모두 불만족이 되지만,
기준이 정한 비교 대상을 쓰면 결과가 달라진다.

## 장기처짐 계수

In [ ]:
rho_prime = np.linspace(0, 0.02, 200)
fig, ax = plt.subplots(figsize=(6.5, 4))
for duration in ["3개월", "6개월", "12개월", "5년이상"]:
    ax.plot(
        rho_prime,
        [
            long_term_deflection_factor(rho_prime=float(r), duration=duration)
            for r in rho_prime
        ],
        label=duration,
    )
ax.set_xlabel("compression steel ratio, rho'")
ax.set_ylabel("lambda_delta")
ax.set_title("Long-term deflection factor")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

압축철근이 있으면 크리프·건조수축에 의한 장기처짐이 줄어든다.

## 균열 제어 (KDS 14 20 20 4.2.3(4))

$$s = 375\left(\frac{\kappa_{cr}}{f_s}\right) - 2.5c_c
\le 300\left(\frac{\kappa_{cr}}{f_s}\right)$$

In [ ]:
fs, s_max, ok = check_crack_control(
    bar_spacing=(400 - 2 * 50) / 3, fy=FY, c_c=50 - 22.2 / 2
)
print(f"철근응력       fs = 2/3*fy = {fs:8.1f} MPa")
print(f"최대 철근 간격 s,max       = {s_max:8.1f} mm")
print(f"배치 철근 간격 s           = {(400 - 2 * 50) / 3:8.1f} mm")
print(f"판정                       = {'만족' if ok else '불만족'}")

In [ ]:
c_c = np.linspace(20, 90, 200)
fig, ax = plt.subplots(figsize=(6.5, 4))
for fy, label in [(400, "SD400"), (500, "SD500"), (600, "SD600")]:
    ax.plot(
        c_c,
        [check_crack_control(bar_spacing=0, fy=fy, c_c=float(c))[1] for c in c_c],
        label=label,
    )
ax.set_xlabel("cover to bar surface, cc (mm)")
ax.set_ylabel("max bar spacing, s (mm)")
ax.set_title("Crack control bar spacing (dry environment)")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## 수축·온도철근 (KDS 14 20 50 4.6.2)

In [ ]:
a_st = shrinkage_temperature_reinforcement(fy=FY, a_g=1000.0 * 200.0)
print(f"수축·온도철근 (t = 200 mm 슬래브, 1 m 폭) = {a_st:.1f} mm^2/m")
print(f"최대 간격                                 = "
      f"{shrinkage_temperature_spacing(thickness=200):.1f} mm")